# TileDB-SOMA Object Operations: Examples

This notebook contains example commands and common usage patterns for opening, inspecting, interacting and working with TileDB-SOMA objects.
This contains only some relevant examples and is not exhaustive.

Use conda env `tiledbsoma`

In [1]:
import tiledbsoma as soma
import tiledbsoma.io
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import s3fs
import os

## Experiments

### Accessing TileDB-SOMA Experiment

Example experiment: `s3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt`

In [2]:
expt_uri = "s3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt"
expt = soma.Experiment.open(expt_uri)
print(expt)

<Experiment 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt' (open for 'r') (2 items)
    'ms': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms' (unopened)
    'obs': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/obs' (unopened)>


### Access various components of the dataset

**Access `obs` (SOMA dataframe with cell level annotations)**

In [3]:
print(expt.obs)
expt.obs.read().concat().to_pandas()

<DataFrame 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/obs' (open for 'r')>


,soma_joinid,obs_id,orig.ident,nFeature_RNA,nCount_RNA,percent.mt,high_mt,high_ct,doublet_score,predicted_doublet,...,donor_treatment_mesh_id,batch,collection,scds,dataset_data_type,curation_dataset_id,dataset_repository,dataset_repository_url,dataset_workflow,sample_number
0,0,GSM3721447_AAACCTGAGCGCCTCA-1,GSM3721447,1502,4678.0,1.859769,False,False,0.098627,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
1,1,GSM3721447_AAACCTGAGGCAAAGA-1,GSM3721447,1156,2891.0,3.977862,False,False,0.093079,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
2,2,GSM3721447_AAACCTGCAGACAAAT-1,GSM3721447,1712,5830.0,2.727273,False,False,0.051482,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
3,3,GSM3721447_AAACCTGCATTCGACA-1,GSM3721447,1358,3972.0,3.172205,False,False,0.056291,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
4,4,GSM3721447_AAACCTGGTCCCGACA-1,GSM3721447,2296,7875.0,3.085714,False,False,0.206497,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102330,102330,GSM3511746_TTCTTAGCATAAGACA-1,GSM3511746,4565,21899.0,2.675921,False,False,0.059406,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
102331,102331,GSM3511746_TTGAACGAGTACTTGC-1,GSM3511746,4947,19979.0,4.089293,False,False,0.109375,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
102332,102332,GSM3511746_TTGACTTTCCGTAGGC-1,GSM3511746,3797,18281.0,1.903616,False,False,0.059406,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
102333,102333,GSM3511746_TTTACTGGTCGCGAAA-1,GSM3511746,4130,19409.0,2.266989,False,False,0.028754,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52


**Access the SOMA Measurement containing the RNA counts data**

In [4]:
expt.ms["RNA"]

<Measurement 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms/RNA' (open for 'r') (6 items)
    'X': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms/RNA/X' (unopened)
    'obsm': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms/RNA/obsm' (unopened)
    'obsp': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms/RNA/obsp' (unopened)
    'uns': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms/RNA/uns' (unopened)
    'var': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms/RNA/var' (unopened)
    'varm': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms/RNA/varm' (unopened)>


**Access `var` (SOMA dataframe with the feature-level annotations)**

In [5]:
expt.ms["RNA"].var.read().concat().to_pandas()

,soma_joinid,var_id,gene_ids,highly_variable,means,dispersions,dispersions_norm,mean,std,genome,feature_types,mt
0,0,MIR1302-2HG,ENSG00000243485,False,6.135343e-05,0.558790,-0.973432,0.000036,0.005923,GRCh38,Gene Expression,False
1,1,FAM138A,ENSG00000237613,False,1.000000e-12,NaN,NaN,0.000000,1.000000,GRCh38,Gene Expression,False
2,2,OR4F5,ENSG00000186092,False,1.000000e-12,NaN,NaN,0.000000,1.000000,GRCh38,Gene Expression,False
3,3,AL627309.1,ENSG00000238009,False,3.780935e-03,1.148042,-0.129381,0.001923,0.047081,GRCh38,Gene Expression,False
4,4,AL627309.3,ENSG00000239945,False,1.062302e-04,1.405674,0.239654,0.000048,0.007914,GRCh38,Gene Expression,False
...,...,...,...,...,...,...,...,...,...,...,...,...
36596,36596,AC141272.1,ENSG00000277836,False,2.542288e-03,2.620404,1.979648,0.000664,0.035817,GRCh38,Gene Expression,False
36597,36597,AC023491.2,ENSG00000278633,False,1.000000e-12,NaN,NaN,0.000000,1.000000,GRCh38,Gene Expression,False
36598,36598,AC007325.1,ENSG00000276017,False,4.141517e-05,0.361767,-1.255651,0.000026,0.004771,GRCh38,Gene Expression,False
36599,36599,AC007325.4,ENSG00000278817,False,1.540291e-03,1.698493,0.659091,0.000672,0.029774,GRCh38,Gene Expression,False


### Create anndata object/h5ad file from TileDB-SOMA experiment

**Create anndata object from TileDB-SOMA experiment**

In [6]:
experiment_path = "s3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt"
expt = soma.Experiment.open(experiment_path)

adata_from_expt = tiledbsoma.io.to_anndata(
    expt,
    measurement_name="RNA"
)

adata_from_expt

AnnData object with n_obs × n_vars = 102335 × 36601
    obs: 'orig.ident', 'nFeature_RNA', 'nCount_RNA', 'percent.mt', 'high_mt', 'high_ct', 'doublet_score', 'predicted_doublet', 'basic_qc', 'leiden', 'sample_id', 'author_cell_id', 'study_id', 'shortname', 'title', 'authors', 'pmid', 'doi', 'publication_date', 'study_description', 'repository', 'repository_identifier', 'repository_url', 'Celltypist_Cell_Annotation', 'Celltypist_Cell_Annotation_Conf_Score', 'scArches_Cell_Annotation', 'scArches_Cell_Annotation_Uncert', 'author_cell_type', 'author_cell_type_cell_ontology_id', 'author_cell_type_cell_ontology_name', 'scArches_Cell_Type_Ontology_ID', 'scArches_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_ID', 'sample_type', 'sample_name', 'sample_cell_line_accession_cellosaurus', 'sample_cell_line_name_cellosaurus', 'sample_tissue', 'sample_tissue_uberon_id', 'sample_tissue_uberon_name', 'sample_pathology', 'sample_treatment', 'sample_treatm

**Create anndata h5ad file from TileDB-SOMA experiment**

`tiledbsoma.io.to_h5ad()` writes the file directly, but it fails when an `obs`/`var` column is entirely `None`/`NaN` — h5py can't serialize an all-null object column as an HDF5 string, raising:

```
TypeError: Can't implicitly convert non-string objects to strings
```

To avoid this, build the AnnData first, coerce object (string) columns to real strings (turning `None` into `""`), then write the h5ad.

In [7]:
experiment_path = "s3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt"
batch = experiment_path.rstrip("/").split("/")[-3]
dataset = experiment_path.rstrip("/").split("/")[-2]
h5ad_name = f"{batch}_{dataset}_anndata.h5ad"
h5ad_path = f"/tmp/{h5ad_name}"

expt = soma.Experiment.open(experiment_path)

# Build the AnnData, then coerce object columns to strings so all-None columns
# (e.g. study_institution) don't break the h5ad write.
adata = tiledbsoma.io.to_anndata(expt, measurement_name="RNA")

def stringify_object_columns(df):
    """Cast object-dtype columns to str, turning None/NaN into empty strings."""
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype("string").fillna("").astype(str)

stringify_object_columns(adata.obs)
stringify_object_columns(adata.var)

adata.write_h5ad(h5ad_path)
print(f"Anndata from {experiment_path} is written to {h5ad_path}")

Anndata from s3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt is written to /tmp/batch1_GSE123813_anndata.h5ad


### Slicing and Filtering Experiments

**Querying for cells with nFeature_RNA > 1000 and genes that are highly_variable**

The query result object (`query`) allows to inspect the query result and selectively access the data instead of loading the entire dataset

In [8]:
query = soma.ExperimentAxisQuery(
    experiment=expt,
    measurement_name="RNA",
    obs_query=soma.AxisQuery(
        value_filter="nFeature_RNA > 1000",
    ),
    var_query=soma.AxisQuery(
        value_filter="highly_variable == True",
    ),
)

**See how many cells and genes are returned with the above query**

In [9]:

(query.n_obs, query.n_vars)

(93535, 2896)

**Access the DataFrame containing the annotations for all cells returned by the above query**

In [10]:
query.obs().concat().to_pandas()

,soma_joinid,obs_id,orig.ident,nFeature_RNA,nCount_RNA,percent.mt,high_mt,high_ct,doublet_score,predicted_doublet,...,donor_treatment_mesh_id,batch,collection,scds,dataset_data_type,curation_dataset_id,dataset_repository,dataset_repository_url,dataset_workflow,sample_number
0,0,GSM3721447_AAACCTGAGCGCCTCA-1,GSM3721447,1502,4678.0,1.859769,False,False,0.098627,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
1,1,GSM3721447_AAACCTGAGGCAAAGA-1,GSM3721447,1156,2891.0,3.977862,False,False,0.093079,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
2,2,GSM3721447_AAACCTGCAGACAAAT-1,GSM3721447,1712,5830.0,2.727273,False,False,0.051482,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
3,3,GSM3721447_AAACCTGCATTCGACA-1,GSM3721447,1358,3972.0,3.172205,False,False,0.056291,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
4,4,GSM3721447_AAACCTGGTCCCGACA-1,GSM3721447,2296,7875.0,3.085714,False,False,0.206497,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93530,102330,GSM3511746_TTCTTAGCATAAGACA-1,GSM3511746,4565,21899.0,2.675921,False,False,0.059406,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
93531,102331,GSM3511746_TTGAACGAGTACTTGC-1,GSM3511746,4947,19979.0,4.089293,False,False,0.109375,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
93532,102332,GSM3511746_TTGACTTTCCGTAGGC-1,GSM3511746,3797,18281.0,1.903616,False,False,0.059406,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52
93533,102333,GSM3511746_TTTACTGGTCGCGAAA-1,GSM3511746,4130,19409.0,2.266989,False,False,0.028754,False,...,NA,batch1,NA,True,GEX,GSE123813,NA,NA,cellranger_7,52


**Load the expression matrix (normalized counts) for the selected cells and genes as a Pandas DataFrame**

In [11]:
query.X(layer_name="data").tables().concat().to_pandas()

,soma_dim_0,soma_dim_1,soma_data
0,0,429,1.143479
1,0,597,2.256611
2,0,1182,2.003235
3,1,1182,1.494927
4,1,1832,1.494927
...,...,...,...
13156144,102334,36043,0.458564
13156145,102334,36094,0.771775
13156146,102334,36366,1.202133
13156147,102334,36371,0.458564


**Load the raw counts**

In [12]:
query.X(layer_name="counts").tables().concat().to_pandas()

,soma_dim_0,soma_dim_1,soma_data
0,0,429,1.0
1,0,597,4.0
2,0,1182,3.0
3,1,1182,1.0
4,1,1832,1.0
...,...,...,...
13156144,102334,36043,1.0
13156145,102334,36094,2.0
13156146,102334,36366,4.0
13156147,102334,36371,1.0


**Querying for CD8+ T cells and genes that are highly_variable**

In [14]:
expt_uri = "s3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt"
expt = soma.Experiment.open(expt_uri)
print(expt)

query = soma.ExperimentAxisQuery(
    experiment=expt,
    measurement_name="RNA",
    obs_query=soma.AxisQuery(
        value_filter="scArches_Cell_Type_Ontology_Name == 'CD8-positive, alpha-beta T cell'",
    ),
    var_query=soma.AxisQuery(
        value_filter="highly_variable == True",
    ),
)

(query.n_obs, query.n_vars)

<Experiment 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt' (open for 'r') (2 items)
    'ms': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/ms' (unopened)
    'obs': 's3://rancho-scrna-live/tiledb-experiments/batch1/GSE123813/tiledbsoma_expt/obs' (unopened)>


(1077, 2896)

**Query result to anndata object**

In [15]:
adata_from_query = query.to_anndata(X_name="data")
adata_from_query

AnnData object with n_obs × n_vars = 1077 × 2896
    obs: 'soma_joinid', 'orig.ident', 'nFeature_RNA', 'nCount_RNA', 'percent.mt', 'high_mt', 'high_ct', 'doublet_score', 'predicted_doublet', 'basic_qc', 'leiden', 'sample_id', 'author_cell_id', 'study_id', 'shortname', 'title', 'authors', 'pmid', 'doi', 'publication_date', 'study_description', 'repository', 'repository_identifier', 'repository_url', 'Celltypist_Cell_Annotation', 'Celltypist_Cell_Annotation_Conf_Score', 'scArches_Cell_Annotation', 'scArches_Cell_Annotation_Uncert', 'author_cell_type', 'author_cell_type_cell_ontology_id', 'author_cell_type_cell_ontology_name', 'scArches_Cell_Type_Ontology_ID', 'scArches_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_ID', 'sample_type', 'sample_name', 'sample_cell_line_accession_cellosaurus', 'sample_cell_line_name_cellosaurus', 'sample_tissue', 'sample_tissue_uberon_id', 'sample_tissue_uberon_name', 'sample_pathology', 'sample_treatment', 's

## Manifests

A **manifest** is a single Parquet file that indexes the cell-level metadata (`obs`) for *every* experiment in a collection. Instead of opening each experiment to inspect its annotations, you can load the manifest into a pandas DataFrame and query across the whole collection at once.

Each row is one cell. The columns are the union of every experiment's `obs` columns, plus four bookkeeping columns added by `tiledbsoma_manifest.py`:

- `experiment` — the source experiment the cell came from
- `collection_name` — the collection the experiment belongs to
- `n_cells` — total cells in that source experiment
- `n_genes` — number of genes (RNA `var`) in that source experiment

Manifests live under `s3://rancho-scrna-live/manifests/`, one per collection (e.g. `Human_10x_manifest.parquet`).

### List the available manifests

In [16]:
fs = s3fs.S3FileSystem()
manifests_base = "s3://rancho-scrna-live/manifests"

for path in fs.ls(manifests_base):
    if path.endswith(".parquet"):
        size_mb = fs.info(path)["size"] / 1e6
        print(f"{path.split('/')[-1]:35s} {size_mb:8.1f} MB")

Cynomolgus_10x_manifest.parquet         12.7 MB
Human_10x_manifest.parquet            1609.7 MB
Human_ss2_manifest.parquet               2.7 MB
Mouse_10x_manifest.parquet             429.3 MB
Mouse_ss2_manifest.parquet               0.6 MB
Rhesus_10x_manifest.parquet              7.7 MB


### Load a manifest

Read the Parquet file straight from S3 into a pandas DataFrame. (The larger manifests such as `Human_10x` are several hundred MB / >1 GB — start with a smaller one like `Human_ss2` to explore.)

In [17]:
manifest_uri = "s3://rancho-scrna-live/manifests/Human_ss2_manifest.parquet"
manifest = pd.read_parquet(manifest_uri)

print(f"{len(manifest):,} cells x {manifest.shape[1]} columns")
print(f"{manifest['experiment'].nunique()} experiments in collection "
      f"'{manifest['collection_name'].iloc[0]}'")
manifest.head()

66,641 cells x 121 columns
23 experiments in collection 'Human_ss2'


,obs_id,orig.ident,nFeature_RNA,nCount_RNA,percent.mt,high_mt,high_ct,predicted_doublet,doublet_score,basic_qc,...,scds,dataset_data_type,curation_dataset_id,dataset_repository,dataset_repository_url,dataset_workflow,experiment,collection_name,n_cells,n_genes
0,11511L&R_11511L-P4-B12,11511L&R,2402,1208875.0,5.110454,False,False,False,0.0,PASS,...,True,GEX,GSE132149,None,None,smartseq,batch10_GSE132149,Human_ss2,2711,36522
1,11511L&R_11511R-P1-C07,11511L&R,2457,1223564.0,6.849662,False,False,False,0.0,PASS,...,True,GEX,GSE132149,None,None,smartseq,batch10_GSE132149,Human_ss2,2711,36522
2,11511L&R_11511R-P1-C12,11511L&R,1819,1181745.0,6.639927,False,False,False,0.0,PASS,...,True,GEX,GSE132149,None,None,smartseq,batch10_GSE132149,Human_ss2,2711,36522
3,11511L&R_11511R-P1-A03,11511L&R,3195,1370480.0,6.610896,False,False,False,0.0,PASS,...,True,GEX,GSE132149,None,None,smartseq,batch10_GSE132149,Human_ss2,2711,36522
4,11511L&R_11511R-P3-G07,11511L&R,2862,1127385.0,8.559365,False,False,False,0.0,PASS,...,True,GEX,GSE132149,None,None,smartseq,batch10_GSE132149,Human_ss2,2711,36522


### See the available annotation columns

The manifest carries all of the per-cell metadata, including donor/sample fields, QC metrics, and the `popv_*` cell-type predictions.

In [18]:
list(manifest.columns)

['obs_id',
 'orig.ident',
 'nFeature_RNA',
 'nCount_RNA',
 'percent.mt',
 'high_mt',
 'high_ct',
 'predicted_doublet',
 'doublet_score',
 'basic_qc',
 'leiden',
 'sample_id',
 'author_cell_id',
 'study_id',
 'shortname',
 'title',
 'authors',
 'pmid',
 'doi',
 'publication_date',
 'study_description',
 'repository',
 'repository_identifier',
 'repository_url',
 'sample_number',
 'scArches_Cell_Annotation',
 'scArches_Cell_Annotation_Uncert',
 'Celltypist_Cell_Annotation',
 'Celltypist_Cell_Annotation_Conf_Score',
 'author_cell_type',
 'author_cell_type_cell_ontology_id',
 'author_cell_type_cell_ontology_name',
 'scArches_Cell_Type_Ontology_ID',
 'scArches_Cell_Type_Ontology_Name',
 'Celltypist_Cell_Type_Ontology_Name',
 'Celltypist_Cell_Type_Ontology_ID',
 'sample_type',
 'sample_name',
 'sample_cell_line_accession_cellosaurus',
 'sample_cell_line_name_cellosaurus',
 'sample_tissue',
 'sample_tissue_uberon_id',
 'sample_tissue_uberon_name',
 'sample_pathology',
 'sample_treatment',
 's

### Per-experiment summary

`n_cells` and `n_genes` are constant within an experiment, so drop duplicates to get a quick overview of what's in the collection.

In [19]:
(manifest[["experiment", "n_cells", "n_genes"]]
    .drop_duplicates()
    .sort_values("n_cells", ascending=False)
    .reset_index(drop=True))

,experiment,n_cells,n_genes
0,batch12_PRJNA591860,20056,36522
1,batch7_PRJEB14362,17264,36522
2,batch13_GSE151091,6394,36522
3,batch7_GSE133707,5957,36522
4,batch5_GSE84465,3134,36601
5,batch9_GSE182870,2840,36522
6,batch10_GSE132149,2711,36522
7,batch7_GSE178209,1908,36522
8,batch10_GSE225331,1010,36522
9,batch10_GSE139079,998,36522


### Explore an annotation column across the whole collection

Because the manifest aggregates every experiment, a single `value_counts()` summarizes the entire collection. Here we look at the consensus cell-type prediction and the tissue of origin.

In [21]:
print("Predicted cell types (popv majority vote):")
print(manifest["scArches_Cell_Type_Ontology_Name"].value_counts().head(15))

print("\nTissues:")
print(manifest["sample_tissue_uberon_name"].value_counts().head(15))

Predicted cell types (popv majority vote):
scArches_Cell_Type_Ontology_Name
embryonic stem cell                       17264
macrophage                                 4943
melanocyte                                 4641
T cell                                     4349
epithelial cell                            4215
retinal rod cell                           4081
fibroblast                                 3000
secretory cell                             2854
CD4-positive, alpha-beta memory T cell     2768
plasma cell                                1717
dendritic cell                             1713
keratinocyte                               1677
malignant cell                             1535
endothelial cell                           1090
basal cell                                  929
Name: count, dtype: int64

Tissues:
sample_tissue_uberon_name
endoderm                                                          17264
lung                                                              1053

### Cross-tabulate: which experiments contain which cell types

A `groupby` / `crosstab` tells you where a given cell population lives *before* you open any experiment.

In [22]:
cell_type = "T cell"

hits = manifest[manifest["scArches_Cell_Type_Ontology_Name"] == cell_type]
print(f"{len(hits):,} '{cell_type}' cells across "
      f"{hits['experiment'].nunique()} experiments\n")
print(hits["experiment"].value_counts())

4,349 'T cell' cells across 3 experiments

experiment
batch12_PRJNA591860    4342
batch8_PRJNA379992        4
batch13_GSE151091         3
Name: count, dtype: int64


### Filter across experiments

Combine any conditions to build a cohort spanning the whole collection — e.g. T cells with more than 1,000 detected genes.

In [23]:
subset = manifest[
    (manifest["scArches_Cell_Type_Ontology_Name"] == "T cell")
    & (manifest["nFeature_RNA"] > 1000)
]

print(f"{len(subset):,} cells matched, from experiments: "
      f"{sorted(subset['experiment'].unique())}")
subset[["experiment", "obs_id", "nFeature_RNA", "nCount_RNA",
        "sample_tissue_uberon_name", "scArches_Cell_Type_Ontology_Name"]].head()

4,135 cells matched, from experiments: ['batch12_PRJNA591860', 'batch13_GSE151091', 'batch8_PRJNA379992']


,experiment,obs_id,nFeature_RNA,nCount_RNA,sample_tissue_uberon_name,scArches_Cell_Type_Ontology_Name
6300,batch12_PRJNA591860,AZ_01_C16_B001546,1766,5.299300e+05,lung,T cell
6303,batch12_PRJNA591860,AZ_01_C13_B001544,3862,1.016561e+06,lung,T cell
6304,batch12_PRJNA591860,AZ_01_E16_B001546,2583,8.085520e+05,lung,T cell
6305,batch12_PRJNA591860,AZ_01_C19_B001544,1726,2.560750e+05,lung,T cell
6314,batch12_PRJNA591860,AZ_01_B18_B001544,3686,5.638610e+05,lung,T cell


### Use the manifest to decide which experiments to open

A common pattern: query the manifest to find the experiments worth loading, then open just those from the collection to pull expression data. This avoids scanning experiments that contain nothing of interest.

In [24]:
collection_name = manifest["collection_name"].iloc[0]
collection_uri = f"s3://rancho-scrna-live/collections/{collection_name}"

# Experiments in this collection that contain the cell type of interest
experiments_of_interest = sorted(
    manifest.loc[manifest["scArches_Cell_Type_Ontology_Name"] == "T cell", "experiment"].unique()
)
print("Experiments to open:", experiments_of_interest)

with soma.Collection.open(collection_uri) as collection:
    for expt_name in experiments_of_interest:
        expt = collection[expt_name]
        query = soma.ExperimentAxisQuery(
            experiment=expt,
            measurement_name="RNA",
            obs_query=soma.AxisQuery(
                value_filter="scArches_Cell_Type_Ontology_Name == 'T cell'"
            ),
        )
        print(f"{expt_name}: {query.n_obs:,} cells x {query.n_vars:,} genes")

Experiments to open: ['batch12_PRJNA591860', 'batch13_GSE151091', 'batch8_PRJNA379992']
batch12_PRJNA591860: 4,342 cells x 36,522 genes
batch13_GSE151091: 3 cells x 36,522 genes
batch8_PRJNA379992: 4 cells x 36,522 genes


## Collections

### List collections in the tiledbsoma_collections directory

In [25]:
fs = s3fs.S3FileSystem()
collections_base = "s3://rancho-scrna-live/collections"
collection_names = [p.split("/")[-1] for p in fs.ls(collections_base)]

for i, name in enumerate(collection_names, start=1):
    collection_uri = f"{collections_base}/{name}"
    try:
        with soma.Collection.open(collection_uri) as collection:
            print(f"{i}. {name}")
    except Exception:
        continue

1. Cynomolgus_10x
2. Human_10x
3. Human_ss2
4. Mouse_10x
5. Mouse_ss2
6. Rhesus_10x


### List experiments available in each collection  

In [26]:
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

fs = s3fs.S3FileSystem()
collections_base = "s3://rancho-scrna-live/collections"
collection_names = [p.split("/")[-1] for p in fs.ls(collections_base)]
experiments = []

for name in collection_names:
    collection_uri = f"{collections_base}/{name}"
    try:
        with soma.Collection.open(collection_uri) as collection:
            for key in collection.keys():
                obj = collection[key]
                if isinstance(obj, soma.Experiment):
                    experiments.append(
                        {
                            "Collection Name": name,
                            "Experiment": key,
                            "Experiment URI": obj.uri,
                        }
                    )
    except Exception:
        continue

experiments = pd.DataFrame(experiments)
print(experiments.to_string(index=False))

Collection Name                                   Experiment                                                                                         Experiment URI
 Cynomolgus_10x                             batch9_CRA002577                             s3://rancho-scrna-live/tiledb-experiments/batch9/CRA002577/tiledbsoma_expt
 Cynomolgus_10x                             batch9_CRA002689                             s3://rancho-scrna-live/tiledb-experiments/batch9/CRA002689/tiledbsoma_expt
 Cynomolgus_10x                             batch9_GSE196792                             s3://rancho-scrna-live/tiledb-experiments/batch9/GSE196792/tiledbsoma_expt
      Human_10x                         batch10_E-MTAB-12339                         s3://rancho-scrna-live/tiledb-experiments/batch10/E-MTAB-12339/tiledbsoma_expt
      Human_10x                         batch10_E-MTAB-13664                         s3://rancho-scrna-live/tiledb-experiments/batch10/E-MTAB-13664/tiledbsoma_expt
      Human_10x 

### Get the number of cells and genes in each experiment in the collections

In [27]:
def summarize_collection(collection_uri, measurement_name="RNA"):
    rows = []
    total_cells = 0

    with soma.Collection.open(collection_uri) as collection:
        for name, expt in collection.items():
            try:
                n_cells = expt.obs.count
                n_genes = expt.ms[measurement_name].var.count

                total_cells += n_cells
                rows.append((name, n_cells, n_genes))

            except Exception as e:
                rows.append((name, f"ERROR: {e}", None))

    df = pd.DataFrame(rows, columns=["experiment", "n_cells", "n_genes"])
    df.index = range(1, len(df) + 1)

    return df, total_cells

collections_base = "s3://rancho-scrna-live/collections"
collections = ["Human_10x", "Human_ss2", "Mouse_10x", "Mouse_ss2"]

for collection in collections:
    print(f"\nCollection: {collection}")
    df, total_cells = summarize_collection(f"{collections_base}/{collection}")
    print(df)
    print(f"Total cells in collection: {total_cells:,}")


Collection: Human_10x
                  experiment  n_cells  n_genes
1       batch10_E-MTAB-12339   133603    36601
2       batch10_E-MTAB-13664   993866    36601
3          batch10_GSE112271    43124    36601
4    batch10_GSE126836_human    80307    36601
5          batch10_GSE136035    24643    36601
..                       ...      ...      ...
529         batch9_GSE249622    87603    36601
530         batch9_GSE249793   135151    36601
531       batch9_PRJNA434002   109543    36601
532       batch9_PRJNA772047     6272    36601
533       batch9_PRJNA932556    35975    36601

[533 rows x 3 columns]
Total cells in collection: 43,773,974

Collection: Human_ss2
                     experiment  n_cells  n_genes
1             batch10_GSE132149     2711    36522
2             batch10_GSE139079      998    36522
3             batch10_GSE225331     1010    36522
4              batch10_GSE86618      470    36522
5             batch12_GSE145825      121    36522
6             batch12_GSE145

### Query on multiple experiments in a collection and create a combined anndata object

**Querying for specific type of cells (e.g., "T cell") in the experiments in Mouse_10x collection**

Run the below cell to define the function and then execute the code in the cell below it to query for a specific cell type across all experiments in the collection and combine results into a single Anndata object. 

In [28]:
import anndata as ad

def query_cell_type_in_collection(collection, cell_type_to_query):
    collection_path = "s3://rancho-scrna-live/collections/" + collection
    collection = soma.Collection.open(collection_path)
    adatas = []

    for name, expt in collection.items():
        obs_df = expt.obs.read().concat().to_pandas()
        cell_type_labels = obs_df["scArches_Cell_Type_Ontology_Name"][(
            obs_df["scArches_Cell_Type_Ontology_Name"]
            .str.contains(cell_type_to_query, na=False)
        )].unique()

        print(f'Dataset {name} \n - Cell types based on query "{cell_type_to_query}": {list(cell_type_labels)}')

        if len(cell_type_labels) > 0:

            with expt.axis_query(
                measurement_name="RNA",
                obs_query=soma.AxisQuery(
                    value_filter=f"scArches_Cell_Type_Ontology_Name in {list(cell_type_labels)}"
                )
            ) as query:
                print(f' - n_obs, n_vars: {query.n_obs} cells, {query.n_vars} vars\n')
                adata = query.to_anndata(X_name="data")
                adata.obs["source_experiment"] = name
                adatas.append(adata)

    combined_adata = ad.concat(adatas, join="outer", label="experiment")
    return combined_adata

Run the below cell to query for a specific cell type across all experiments in the collection and combine results into a single Anndata object. You can update the collection and the cell type you want to query for as needed.

In [29]:
collection = 'Mouse_10x'
cell_type_to_query = 'T cell'
combined_adata = query_cell_type_in_collection(collection, cell_type_to_query)
combined_adata

Dataset batch10_CRA010641 
 - Cell types based on query "T cell": ['mature NK T cell', 'CD4-positive, alpha-beta T cell', 'T cell', 'CD8-positive, alpha-beta T cell', 'regulatory T cell', 'naive T cell']
 - n_obs, n_vars: 36640 cells, 32285 vars

Dataset batch10_GSE126836_mouse 
 - Cell types based on query "T cell": []
Dataset batch10_GSE150742 
 - Cell types based on query "T cell": ['T cell', 'mature NK T cell', 'CD8-positive, alpha-beta T cell', 'CD4-positive, alpha-beta T cell', 'regulatory T cell']
 - n_obs, n_vars: 1989 cells, 32285 vars

Dataset batch10_GSE164674 
 - Cell types based on query "T cell": []
Dataset batch10_GSE175546 
 - Cell types based on query "T cell": ['CD8-positive, alpha-beta T cell', 'T cell', 'mature NK T cell']
 - n_obs, n_vars: 1483 cells, 32285 vars

Dataset batch10_GSE179501 
 - Cell types based on query "T cell": ['mature NK T cell', 'CD4-positive, alpha-beta T cell', 'T cell', 'CD8-positive, alpha-beta T cell', 'regulatory T cell']
 - n_obs, n_vars:

AnnData object with n_obs × n_vars = 1435782 × 32285
    obs: 'soma_joinid', 'orig.ident', 'nFeature_RNA', 'nCount_RNA', 'percent.mt', 'high_mt', 'high_ct', 'doublet_score', 'predicted_doublet', 'basic_qc', 'leiden', 'sample_id', 'author_cell_id', 'study_id', 'shortname', 'title', 'authors', 'pmid', 'doi', 'publication_date', 'study_description', 'repository', 'repository_identifier', 'repository_url', 'sample_number', 'Celltypist_Cell_Annotation', 'Celltypist_Cell_Annotation_Conf_Score', 'scArches_Cell_Annotation', 'scArches_Cell_Annotation_Uncert', 'author_cell_type', 'author_cell_type_cell_ontology_id', 'author_cell_type_cell_ontology_name', 'scArches_Cell_Type_Ontology_ID', 'scArches_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_ID', 'sample_type', 'sample_name', 'sample_cell_line_accession_cellosaurus', 'sample_cell_line_name_cellosaurus', 'sample_tissue', 'sample_tissue_uberon_id', 'sample_tissue_uberon_name', 'sample_pathology', '

### Running the downstream pipeline with the query result

In [30]:
sc.pp.normalize_total(combined_adata, target_sum=1, exclude_highly_expressed=True, inplace=True)    # Normalization
print(combined_adata.X)

/home/anne.cooley/conda/envs/tiledbsoma/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3150921988 stored elements and shape (1435782, 32285)>
  Coords	Values
  (0, 16)	0.03350786119699478
  (0, 20)	0.056754183024168015
  (0, 50)	0.03350786119699478
  (0, 52)	0.03350786119699478
  (0, 60)	0.0476527214050293
  (0, 63)	0.0476527214050293
  (0, 115)	0.03350786119699478
  (0, 118)	0.03350786119699478
  (0, 156)	0.0476527214050293
  (0, 178)	0.0476527214050293
  (0, 179)	0.0476527214050293
  (0, 193)	0.0476527214050293
  (0, 197)	0.03350786119699478
  (0, 198)	0.03350786119699478
  (0, 203)	0.03350786119699478
  (0, 206)	0.03350786119699478
  (0, 208)	0.03350786119699478
  (0, 210)	0.0476527214050293
  (0, 234)	0.0476527214050293
  (0, 238)	0.03350786119699478
  (0, 244)	0.03350786119699478
  (0, 326)	0.08030854165554047
  (0, 327)	0.03350786119699478
  (0, 334)	0.0476527214050293
  (0, 346)	0.06347811222076416
  :	:
  (1435781, 30027)	1.0
  (1435781, 30053)	1.2421947717666626
  (1435781, 30196)	1.3868573904037476
 

**PCA plotting**

In [ ]:
sc.pp.pca(combined_adata)                                               # PCA computation
sc.pl.pca(combined_adata, color="scArches_Cell_Type_Ontology_Name")        # Scatter plot in PCA coordinates

**UMAP plotting**

In [ ]:
sc.pp.neighbors(combined_adata)
sc.tl.umap(combined_adata)
sc.pl.umap(combined_adata, color="scArches_Cell_Type_Ontology_Name")

In [33]:
combined_adata

AnnData object with n_obs × n_vars = 1435782 × 32285
    obs: 'soma_joinid', 'orig.ident', 'nFeature_RNA', 'nCount_RNA', 'percent.mt', 'high_mt', 'high_ct', 'doublet_score', 'predicted_doublet', 'basic_qc', 'leiden', 'sample_id', 'author_cell_id', 'study_id', 'shortname', 'title', 'authors', 'pmid', 'doi', 'publication_date', 'study_description', 'repository', 'repository_identifier', 'repository_url', 'sample_number', 'Celltypist_Cell_Annotation', 'Celltypist_Cell_Annotation_Conf_Score', 'scArches_Cell_Annotation', 'scArches_Cell_Annotation_Uncert', 'author_cell_type', 'author_cell_type_cell_ontology_id', 'author_cell_type_cell_ontology_name', 'scArches_Cell_Type_Ontology_ID', 'scArches_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_ID', 'sample_type', 'sample_name', 'sample_cell_line_accession_cellosaurus', 'sample_cell_line_name_cellosaurus', 'sample_tissue', 'sample_tissue_uberon_id', 'sample_tissue_uberon_name', 'sample_pathology', '

In [ ]:
sc.pp.log1p(combined_adata)                                             # Logarithmic transformation
sc.pp.highly_variable_genes(combined_adata)                             # Annotate highly variable genes
sc.pp.filter_cells(combined_adata, min_genes=3)                         # Filter out cells with less than 3 genes expressed
combined_adata

**Querying for specific type of cells (e.g., "T cell") and nFeature_RNA > 500, in the experiments in Mouse_10x collection**

Run the below cell to define the function and then execute the code in the cell below it to query for a specific cell type across all experiments in the collection and combine results into a single Anndata object.

In [36]:
def query_based_on_cell_and_gene_in_collection(collection, cell_type_to_query, gene_parameter, min_features=0):
    collection_path = "s3://rancho-scrna-live/collections/" + collection
    collection = soma.Collection.open(collection_path)
    adatas = []

    for name, expt in collection.items():
        obs_df = expt.obs.read().concat().to_pandas()
        cell_type_labels = obs_df["scArches_Cell_Type_Ontology_Name"][(
            obs_df["scArches_Cell_Type_Ontology_Name"]
            .str.contains(cell_type_to_query, na=False)
        )].unique()

        print(f'Dataset {name} \n - Cell types based on query "{cell_type_to_query}": {list(cell_type_labels)}')

        if len(cell_type_labels) > 0:

            value_filter = (
                f"scArches_Cell_Type_Ontology_Name in {list(cell_type_labels)} "
                f"and {gene_parameter} > {min_features}"
            )

            with expt.axis_query(
                measurement_name="RNA",
                obs_query=soma.AxisQuery(
                    value_filter=value_filter
                )
            ) as query:
                print(f' - n_obs, n_vars: {query.n_obs} cells, {query.n_vars} vars\n')
                adata = query.to_anndata(X_name="data")
                adata.obs["source_experiment"] = name
                adatas.append(adata)

    combined_adata_2 = ad.concat(adatas, join="outer", label="experiment")
    return combined_adata_2

Run the below cell to query for a specific cell type across all experiments in the collection and combine results into a single Anndata object. You can update the collection and the cell type you want to query for as needed.

In [37]:
collection = 'Mouse_10x'
cell_type_to_query = 'T cell'
gene_parameter = 'nFeature_RNA'
min_features = 500
combined_adata2 = query_based_on_cell_and_gene_in_collection(collection, cell_type_to_query, gene_parameter, min_features)
combined_adata2

Dataset batch10_CRA010641 
 - Cell types based on query "T cell": ['mature NK T cell', 'CD4-positive, alpha-beta T cell', 'T cell', 'CD8-positive, alpha-beta T cell', 'regulatory T cell', 'naive T cell']
 - n_obs, n_vars: 36308 cells, 32285 vars

Dataset batch10_GSE126836_mouse 
 - Cell types based on query "T cell": []
Dataset batch10_GSE150742 
 - Cell types based on query "T cell": ['T cell', 'mature NK T cell', 'CD8-positive, alpha-beta T cell', 'CD4-positive, alpha-beta T cell', 'regulatory T cell']
 - n_obs, n_vars: 1986 cells, 32285 vars

Dataset batch10_GSE164674 
 - Cell types based on query "T cell": []
Dataset batch10_GSE175546 
 - Cell types based on query "T cell": ['CD8-positive, alpha-beta T cell', 'T cell', 'mature NK T cell']
 - n_obs, n_vars: 1480 cells, 32285 vars

Dataset batch10_GSE179501 
 - Cell types based on query "T cell": ['mature NK T cell', 'CD4-positive, alpha-beta T cell', 'T cell', 'CD8-positive, alpha-beta T cell', 'regulatory T cell']
 - n_obs, n_vars:

AnnData object with n_obs × n_vars = 1403117 × 32285
    obs: 'soma_joinid', 'orig.ident', 'nFeature_RNA', 'nCount_RNA', 'percent.mt', 'high_mt', 'high_ct', 'doublet_score', 'predicted_doublet', 'basic_qc', 'leiden', 'sample_id', 'author_cell_id', 'study_id', 'shortname', 'title', 'authors', 'pmid', 'doi', 'publication_date', 'study_description', 'repository', 'repository_identifier', 'repository_url', 'sample_number', 'Celltypist_Cell_Annotation', 'Celltypist_Cell_Annotation_Conf_Score', 'scArches_Cell_Annotation', 'scArches_Cell_Annotation_Uncert', 'author_cell_type', 'author_cell_type_cell_ontology_id', 'author_cell_type_cell_ontology_name', 'scArches_Cell_Type_Ontology_ID', 'scArches_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_Name', 'Celltypist_Cell_Type_Ontology_ID', 'sample_type', 'sample_name', 'sample_cell_line_accession_cellosaurus', 'sample_cell_line_name_cellosaurus', 'sample_tissue', 'sample_tissue_uberon_id', 'sample_tissue_uberon_name', 'sample_pathology', '

### Find experiments that express a particular gene

The cell-type searches above filter `obs` (cell annotations). A gene's expression is not in `obs` — it lives in the `X` matrix, indexed by `var` (the gene table). To search by gene you select it with a **`var_query`** instead of an `obs_query`, then read the counts.

The function below loops over every experiment in a collection and, for each one:

1. selects the gene on the `var` axis (`var_id` is the gene symbol; use `gene_ids` for Ensembl IDs);
2. reports whether the gene is measured at all (`n_vars == 0` means it isn't in that experiment's `var`);
3. reads the `counts` layer for that single gene and counts how many cells express it above `min_count`.

It returns a summary DataFrame across the collection.

In [35]:
def find_experiments_expressing_gene(collection, gene, min_count=0, gene_column="var_id"):
    """Report, per experiment in a collection, how many cells express `gene`."""
    collection_path = "s3://rancho-scrna-live/collections/" + collection
    rows = []

    with soma.Collection.open(collection_path) as coll:
        for name, expt in coll.items():
            with expt.axis_query(
                measurement_name="RNA",
                var_query=soma.AxisQuery(value_filter=f"{gene_column} == '{gene}'"),
            ) as query:
                if query.n_vars == 0:
                    print(f"{name}: gene '{gene}' not measured")
                    rows.append({"experiment": name, "measured": False,
                                 "n_expressing": 0, "n_cells": query.n_obs, "pct": 0.0})
                    continue

                # Long-form table of nonzero entries for this one gene: soma_data = count
                x = query.X("counts").tables().concat().to_pandas()
                n_expressing = int((x["soma_data"] > min_count).sum())
                pct = 100 * n_expressing / query.n_obs if query.n_obs else 0.0
                print(f"{name}: {n_expressing:,}/{query.n_obs:,} cells "
                      f"express {gene} (> {min_count})  [{pct:.1f}%]")
                rows.append({"experiment": name, "measured": True,
                             "n_expressing": n_expressing, "n_cells": query.n_obs,
                             "pct": round(pct, 1)})

    return pd.DataFrame(rows).sort_values("n_expressing", ascending=False).reset_index(drop=True)

Run it across a collection — e.g. `CD8A` in `Human_ss2`:

In [38]:
gene_summary = find_experiments_expressing_gene("Human_10x", gene="KISS1", min_count=0)
gene_summary

batch10_E-MTAB-12339: 5,039/133,603 cells express KISS1 (> 0)  [3.8%]
batch10_E-MTAB-13664: 9,611/993,866 cells express KISS1 (> 0)  [1.0%]
batch10_GSE112271: 286/43,124 cells express KISS1 (> 0)  [0.7%]
batch10_GSE126836_human: 6/80,307 cells express KISS1 (> 0)  [0.0%]
batch10_GSE136035: 2/24,643 cells express KISS1 (> 0)  [0.0%]
batch10_GSE137829: 1,279/49,013 cells express KISS1 (> 0)  [2.6%]
batch10_GSE141578: 0/23,728 cells express KISS1 (> 0)  [0.0%]
batch10_GSE143706: 1,303/37,276 cells express KISS1 (> 0)  [3.5%]
batch10_GSE148434: 6/37,528 cells express KISS1 (> 0)  [0.0%]
batch10_GSE150290: 1,839/219,463 cells express KISS1 (> 0)  [0.8%]
batch10_GSE153765: 0/8,975 cells express KISS1 (> 0)  [0.0%]
batch10_GSE161192: 0/6,178 cells express KISS1 (> 0)  [0.0%]
batch10_GSE163558: 127/45,396 cells express KISS1 (> 0)  [0.3%]
batch10_GSE164690: 156/138,069 cells express KISS1 (> 0)  [0.1%]
batch10_GSE168668: 254/12,176 cells express KISS1 (> 0)  [2.1%]
batch10_GSE168933: 12/44,896

,experiment,measured,n_expressing,n_cells,pct
0,batch13_E-MTAB-8210,True,10993,107439,10.2
1,batch11_GSE226225,True,10592,46087,23.0
2,batch10_E-MTAB-13664,True,9611,993866,1.0
3,batch9_E-MTAB-11267,True,7676,60141,12.8
4,batch10_E-MTAB-12339,True,5039,133603,3.8
...,...,...,...,...,...
528,batch10_GSE141578,True,0,23728,0.0
529,batch9_GSE249622,True,0,87603,0.0
530,batch10_GSE161192,True,0,6178,0.0
531,batch9_PRJNA772047,True,0,6272,0.0


To pull the actual cells expressing a gene into an AnnData object (rather than just counting them), combine a `var_query` for the gene with the `obs_query`/`X` pattern used earlier — for example, restrict to a cell type *and* a gene, then `query.to_anndata(X_name="data")` per experiment and `ad.concat(...)` the results, exactly as `query_cell_type_in_collection` does.